# ICD ReAct v2 Tool Playground

Use this notebook to test the first ICD ReAct v2 keyword-exploration tool pair. The first call lists top-level Alphabetic Index headings for one letter, and the second call opens one selected heading into its full nested hierarchy.

In [20]:
from __future__ import annotations

import importlib
import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repo root containing pyproject.toml.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import baselines.icd_react_v2 as icd_react_v2
import baselines.icd_react_v2.tools as icd_react_v2_tools

importlib.reload(icd_react_v2_tools)
icd_react_v2 = importlib.reload(icd_react_v2)

load_config = icd_react_v2.load_config
list_guideline_toc = icd_react_v2.list_guideline_toc
list_index_letter_headings = icd_react_v2.list_index_letter_headings
list_tabular_chapters = icd_react_v2.list_tabular_chapters
open_guideline_section = icd_react_v2.open_guideline_section
open_index_heading_hierarchy = icd_react_v2.open_index_heading_hierarchy
open_tabular_chapter = icd_react_v2.open_tabular_chapter
open_tabular_block = icd_react_v2.open_tabular_block
open_tabular_code = icd_react_v2.open_tabular_code

In [21]:
config = load_config()
summary = {
    "execution_env": config.execution_env,
    "manuals_root": str(config.manuals_root),
    "index_xml_path": str(config.index_xml_path),
    "tabular_xml_path": str(config.tabular_xml_path),
}
print(json.dumps(summary, indent=2))

{
  "execution_env": "local",
  "manuals_root": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\data\\ICD-2019-manual",
  "index_xml_path": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\data\\ICD-2019-manual\\icd10cm_tabular_2019\\icd10cm_index_2019.xml",
  "tabular_xml_path": "C:\\Users\\sonutka\\main\\1-Projects\\manual_deterministic_executor\\data\\ICD-2019-manual\\icd10cm_tabular_2019\\icd10cm_tabular_2019.xml"
}


## 1. List Top-Level Headings for A

This is the top browse step. It returns the main Alphabetic Index headings under one starting letter without opening the nested hierarchy yet. Use `start_index` with `limit` to page through large buckets.

In [ ]:
a_headings = list_index_letter_headings(letter="D", limit=None, start_index=0, config=config)
preview = {
    "letter": a_headings["letter"],
    "start_index": a_headings["start_index"],
    "limit": a_headings["limit"],
    "count": a_headings["count"],
    "results": a_headings["results"][:10],
}
print(json.dumps(preview, indent=2))

## 2. Open One Heading Hierarchy

Pick one `entry_id` from the previous cell and inspect its nested child terms.

In [ ]:
entry_id = a_headings["results"][7]["entry_id"]
heading_hierarchy = open_index_heading_hierarchy(entry_id=entry_id, config=config)
print(json.dumps(heading_hierarchy, indent=2)[:12000])

In [ ]:
tabular_chapters = list_tabular_chapters(config=config)
chapter_preview = {
    "chapter_count": tabular_chapters["chapter_count"],
    "chapters": tabular_chapters["chapters"],
}
print(json.dumps(chapter_preview, indent=2)[:12000])

## 4. Open One Tabular Chapter

Pick one `chapter_id` from the previous cell and inspect its blocks only. This step should stop at rows such as `Intestinal infectious diseases (A00-A09)`.

In [ ]:
chapter_id = tabular_chapters["chapters"][0]["chapter_id"]
tabular_chapter = open_tabular_chapter(chapter_id=chapter_id, config=config)
print(json.dumps(tabular_chapter, indent=2)[:12000])

## 5. Open One Tabular Block

Pick one `section_id` from the chapter opener and inspect the direct codes beneath that block.

In [ ]:
section_id = tabular_chapter["blocks"][0]["section_id"]
tabular_block = open_tabular_block(section_id=section_id, config=config)
print(json.dumps(tabular_block, indent=2)[:12000])

## 6. Open One Tabular Code

Take one exact code from the block opener and inspect the full Tabular entry.

In [ ]:
tabular_code = tabular_block["codes"][0]["code"]
tabular_entry = open_tabular_code(code=tabular_code, config=config)
print(json.dumps(tabular_entry, indent=2)[:12000])

In [22]:
guideline_toc = list_guideline_toc(limit=None, config=config)
guideline_title_filter = list_guideline_toc(section_prefix="b. Tabular", limit=None, config=config)
guideline_preview = {
    "count": guideline_toc["count"],
    "results": guideline_toc["results"],
    "filtered_results": guideline_title_filter["results"],
}
print(json.dumps(guideline_preview, indent=2))

{
  "count": 212,
  "results": [
    {
      "section_id": "guidelines|8|section-i-conventions-general-coding-guidelines-and-chapter-specific-guidelines",
      "title": "Section I. Conventions, general coding guidelines and chapter specific guidelines",
      "level": 1
    },
    {
      "section_id": "guidelines|8|a-conventions-for-the-icd-10-cm",
      "title": "A. Conventions for the ICD-10-CM",
      "level": 2
    },
    {
      "section_id": "guidelines|8|1-the-alphabetic-index-and-tabular-list",
      "title": "1. The Alphabetic Index and Tabular List",
      "level": 3
    },
    {
      "section_id": "guidelines|8|2-format-and-structure",
      "title": "2. Format and Structure:",
      "level": 3
    },
    {
      "section_id": "guidelines|8|3-use-of-codes-for-reporting-purposes",
      "title": "3. Use of codes for reporting purposes",
      "level": 3
    },
    {
      "section_id": "guidelines|8|4-placeholder-character",
      "title": "4. Placeholder character",
     

## 7. Open One Guidelines Section

Pick one `section_id` from the guidelines TOC and inspect the extracted rule text.

In [23]:
# target_guideline = next(
#     (item for item in guideline_toc["results"] if item["title"].startswith("b. Tabular List abbreviations")),
#     guideline_toc["results"][0],
# )
guideline_section_id = "guidelines|21|c-chapter-specific-coding-guidelines"
guideline_section = open_guideline_section(section_id=guideline_section_id, config=config)
guideline_section_preview = {
    "section_id": guideline_section["section_id"],
    "title": guideline_section["title"],
    "level": guideline_section["level"],
    "page_start": guideline_section["page_start"],
    "page_end": guideline_section["page_end"],
    "text_preview": guideline_section["text"],
}
print(json.dumps(guideline_section_preview, indent=2))

{
  "section_id": "guidelines|21|c-chapter-specific-coding-guidelines",
  "title": "C. Chapter-Specific Coding Guidelines",
  "level": 2,
  "page_start": 21,
  "page_end": 107,
  "text_preview": "C. Chapter-Specific Coding Guidelines\nIn addition to general coding guidelines, there are guidelines for specific diagnoses\nand/or conditions in the classification. Unless otherwise indicated, these guidelines\napply to all health care settings. Please refer to Section II for guidelines on the\nselection of principal diagnosis.\n1. Chapter 1: Certain Infectious and Parasitic Diseases (A00-B99)\na. Human Immunodeficiency Virus (HIV) Infections\n1)\nCode only confirmed cases\nCode only confirmed cases of HIV infection/illness. This is an\nexception to the hospital inpatient guideline Section II, H.\nIn this context, \"confirmation\" does not require documentation\nof positive serology or culture for HIV; the provider's\ndiagnostic statement that the patient is HIV positive, or has an\nHIV-rela

In [ ]:
print(guideline_section["text"])